Put conditions for determining blocked rxns into [FVA](#FVA)

In [ ]:
# For reducing the model's size to simplify building it. Optional, but make sure you know all potential conditions you'll be testing.
import cobra,json,os,sys,re,pandas as pd, io, requests
from collections import OrderedDict
from copy import deepcopy

# keep going up until the "pycore" folder is found
wd = os.getcwd()
os.chdir('..')
base_path = os.getcwd()
sys.path.append(base_path)
from pycore import *

os.chdir(wd)

/Users/ejm6426/Documents/pkRBA/pycore/../build_model/input/iIsor850_RBA.json


In [2]:
# example of how you'd add all products considered in max yield calculations, in case they reduce the number of blocked rxns; to use, modify SM_path as needed and uncomment the following lines. Note that this will add all products to the model, which may not be desired for your application.
SM_path = '../application/input/iIsor850_RBA_pathwayadd.json'
model = load_cobra_model(SM_path)

for r in model.reactions:
	for note in ('turned_off_for_RBA','altered_for_RBA'):
		if getattr(r, 'notes', {}).get(note, False):
			# set lower and upper bounds to "Original bounds were (x,y)"
			original_bounds = re.search(r'Original bounds were \(([^,]*),([^)]*)\)', r.notes.get(note,''))
			if original_bounds:
				r.lower_bound = float(original_bounds.group(1))
				r.upper_bound = float(original_bounds.group(2))
				print(f"Re-enabled {r.id} for RBA model building, with bounds ({r.lower_bound},{r.upper_bound})")
			del r.notes[note]

# yield-specific changes (only when they expand the bounds) also applied here for consistency
model.reactions.ACONTb_c.upper_bound = 1000
model.reactions.Biomass_MFA_succ_with_MFA_extras_with_GAM.upper_bound = 1000
model.reactions.Biomass_MFA_succ_with_MFA_extras.upper_bound = 1000
model.reactions.Biomass_MFA_succ_macros_only.upper_bound = 1000
model.reactions.Biomass_MFA_succ_raw.upper_bound = 1000
model.reactions.Biomass_MFA_succ_raw_normalized.upper_bound = 1000
model.reactions.Biomass_Isor.upper_bound = 1000

Re-enabled ARABDt_c for RBA model building, with bounds (-1000.0,1000.0)
Re-enabled EX_arab__D_e for RBA model building, with bounds (0.0,1000.0)
Re-enabled EX_csn_e for RBA model building, with bounds (0.0,1000.0)
Re-enabled CYTK2_c for RBA model building, with bounds (-1000.0,1000.0)
Re-enabled EX_cytd_e for RBA model building, with bounds (0.0,1000.0)
Re-enabled EX_fru_e for RBA model building, with bounds (0.0,1000.0)
Re-enabled E4Pt_c_m for RBA model building, with bounds (-1000.0,1000.0)
Re-enabled EX_nmn_e for RBA model building, with bounds (0.0,1000.0)
Re-enabled NCAMt_c_n for RBA model building, with bounds (-1000.0,1000.0)
Re-enabled NADPHt_c_r for RBA model building, with bounds (-1000.0,1000.0)
Re-enabled EX_dad_2_e for RBA model building, with bounds (0.0,1000.0)
Re-enabled MI1Pt_c_r for RBA model building, with bounds (-1000.0,1000.0)
Re-enabled TYMR_c for RBA model building, with bounds (-1000.0,1000.0)
Re-enabled EX_duri_e for RBA model building, with bounds (0.0,1000.

In [3]:
# find metabolites only used in one reaction
removed_rxns = 0
remove_orphans = True
# set min_rxns_per_met to 1 to keep all metabolites that can be used in the model, even if they are only used in 1 rxn, since they may be used in the PTM stoichiometry for proteins in RBA.
# set min_rxns_per_met to 2 to remove all metabolites that can't be used in the model, since they can't be mass-balanced without the flux thru their 1 rxn being 0
min_rxns_per_met = 1
# add option to determine whether to add demand reactions for orphan metabolites or just remove them from the model
# set to True for building a precursor to an RBA model, since PTMs may function as sinks for metabolites; set to False to remove metabolites the GSM model can't use
add_demand_reactions = True
# reduce reversible rxns to irreversible if FVA shows they can only go in one direction, to reduce the scope of the model and improve solving time
# not recommended if you have evidence of cycling within a specific rxn (e.g., MFA data showing both directions of a rxn are used simultaneously), since this will remove the ability to cycle through that rxn in the RBA model. Can be important for representing heat-generating processes, for example.
# set to True to enable this feature; set to False to disable it
reduce_reversible_rxns = True

# minimum flux distinguishable from zero, used to determine blocked reactions and whether reversible rxns can only go in one direction. This is set to the model's tolerance, which is usually 1e-6, but can be changed if needed.
flux_tolerance = model.tolerance

# remove all rxns that are not used anymore
for rxn in model.reactions.copy():
	if rxn.bounds == (0, 0):
		# print(f'Removed reaction {rxn.id} with zero bounds')
		model.remove_reactions(rxn, remove_orphans=remove_orphans)
	elif len(rxn.metabolites) == 0:
		# print(f'Removed reaction {rxn.id} with no metabolites')
		model.remove_reactions(rxn, remove_orphans=remove_orphans)

/Applications/anaconda3/envs/rba/lib/python3.9/site-packages/cobra/core/model.py:777: UserWarning: need to pass in a list
  warn("need to pass in a list")


In [4]:
# find all metabolites that are potenially identical
# merge them into one metabolite
metabolite_dict = {}
# metabolite_dict_essentials is for formulas, charges, compartments
metabolite_dict_essentials = {}
for met in model.metabolites:
	met_key = (met.formula, met.charge, met.compartment, tuple(sorted(met.notes.items())), tuple(sorted(met.annotation.items())))
	if met_key in metabolite_dict:
		print(f'Consider merging metabolite {met.id} into {metabolite_dict[met_key].id}')
		# merge met into metabolite_dict[met_key]
		met_to_keep = metabolite_dict[met_key]
		# for rxn in met.reactions:
		# 	rxn.add_metabolites({met_to_keep: rxn.metabolites[met]}, combine=True)
		# 	rxn.remove_metabolites({met: rxn.metabolites[met]})
		# model.metabolites.remove(met)
	elif (met.formula, met.charge, met.compartment) in metabolite_dict_essentials:
		# check if its notes and annotations are a subset of any existing metabolite with the same formula, charge, and compartment
		existing_met = metabolite_dict_essentials[(met.formula, met.charge, met.compartment)]
		if (tuple(sorted(met.notes.items())), tuple(sorted(met.annotation.items()))) <= (tuple(sorted(existing_met.notes.items())), tuple(sorted(existing_met.annotation.items()))):
			print(f'Consider merging metabolite {met.id} into {existing_met.id}')
			# merge met into existing_met
			met_to_keep = existing_met
			# for rxn in met.reactions:
			# 	rxn.add_metabolites({met_to_keep: rxn.metabolites[met]}, combine=True)
			# 	rxn.remove_metabolites({met: rxn.metabolites[met]})
			# model.metabolites.remove(met)
	else:
		# check if the metabolite's formula, charge, and compartment are unique
		metabolite_dict[met_key] = met

Consider merging metabolite epist181_rm into lanost161_rm
Consider merging metabolite fecost161_rm into epist161_rm
Consider merging metabolite fecost181_l into epist181_l
Consider merging metabolite lanost161_l into epist181_l
Consider merging metabolite fecost181_rm into lanost161_rm
Consider merging metabolite epist161_l into fecost161_l
Consider merging metabolite 3mlcl_mm into 1mlcl_mm
Consider merging metabolite 3hpp_c into 3oxpp_c
Consider merging metabolite 4hbz_e into 3hpp_e
Consider merging metabolite amorph_c into 3oxpp_c
Consider merging metabolite nhemrd_c into 3oxpp_c
Consider merging metabolite arte_c into 3oxpp_c
Consider merging metabolite nhemox_c into 3oxpp_c
Consider merging metabolite arte_e into 3hpp_e
Consider merging metabolite citm_e into 3hpp_e
Consider merging metabolite cou_c into 3oxpp_c
Consider merging metabolite cou_e into 3hpp_e
Consider merging metabolite itacon_c into 3oxpp_c
Consider merging metabolite itacon_e into 3hpp_e
Consider merging metabolite

In [5]:
# find all rxns that are identical except for their names and ids

# tests to ensure deletion of identical reactions works
## duplicate the 1st reaction under a different id and name
reference_rxn = model.reactions[0]
duplicate_rxn = reference_rxn.copy()
duplicate_rxn.id = 'duplicate_rxn'
duplicate_rxn.name = 'Duplicate Reaction'
reversed_rxn = reference_rxn.copy()
reversed_rxn.id = 'reversed_rxn'
reversed_rxn.name = 'Reversed Reaction'
reversed_metabolites = {met.id: -coef for met, coef in reference_rxn.metabolites.items()}
reversed_rxn.add_metabolites(reversed_metabolites, combine=False)
reversed_rxn.bounds = (-reference_rxn.bounds[1], -reference_rxn.bounds[0])
model.add_reactions([reversed_rxn, duplicate_rxn])

rxn_dict = {}
for rxn in model.reactions.copy():
	rxn_key = (frozenset(rxn.metabolites.items()), rxn.bounds)
	if rxn_key in rxn_dict:
		rxn_dict[rxn_key].append(rxn)
	# check for rxns with opposite stoichiometry and bounds
	elif (frozenset({met: -coef for met, coef in rxn.metabolites.items()}.items()), (-rxn.bounds[1], -rxn.bounds[0])) in rxn_dict:
		# if the reaction is the opposite of an existing one, remove it
		existing_rxn = rxn_dict[(frozenset({met: -coef for met, coef in rxn.metabolites.items()}.items()), (-rxn.bounds[1], -rxn.bounds[0]))][0]
		print(f'Removed reaction {rxn.id} identical to {existing_rxn.id} with opposite stoichiometry and bounds')
		model.remove_reactions([rxn], remove_orphans=remove_orphans)
		# m.reactions.remove(rxn)
		continue
	else:
		rxn_dict[rxn_key] = [rxn]
# remove all but one of the identical rxns
for rxn_list in rxn_dict.values():
	if len(rxn_list) > 1:
		for rxn in rxn_list[1:]:
			print(f'Removed reaction {rxn.id} identical to {rxn_list[0].id}')
			model.remove_reactions([rxn], remove_orphans=remove_orphans)

Removed reaction reversed_rxn identical to lumpFACS160_c with opposite stoichiometry and bounds
Removed reaction duplicate_rxn identical to lumpFACS160_c
Removed reaction Biomass_MFA_succ_with_MFA_extras_with_GAM identical to Biomass_MFA
Removed reaction 3HPPB3 identical to 3HPPA2
Removed reaction 3HPPB4 identical to 3HPPA3


In [6]:
# iteratively remove rxns until no more can be removed
c = 0
while removed_rxns == 0:
	c += 1
	removed_rxns = 0
	removed_mets = 0
	for met in model.metabolites.copy():
		if len(met.reactions) != 0 and len(met.reactions) < min_rxns_per_met:
			print(f'Removed reaction {met.reactions} using metabolite {met}')
			model.remove_reactions(list(met.reactions)[0], remove_orphans=remove_orphans)
			# model.metabolites.remove(met)
			removed_rxns += 1
			removed_mets += 1
		elif len(met.reactions) == 0:
			# remove metabolite with no reactions
			print(f'Removed metabolite {met} with no reactions')
			model.remove_metabolites([met])
			removed_mets += 1
	# if no rxns were removed on 1st iteration, break the loop
	if removed_rxns == 0 and c == 1:
		print('No reactions removed on 1st iteration, breaking loop')
		break

No reactions removed on 1st iteration, breaking loop


### FVA

In [7]:
# remove all blocked reactions (determined via flux variability analysis)
original_bounds = {rxn.id: rxn.bounds for rxn in model.reactions}
with model as m:
	# add demand rxns if allowed
	if add_demand_reactions:
		for met in m.metabolites.copy():
			m.add_boundary(met, type='demand')
	# turn on all secretions (to account for product yield simulations) and uptakes (to account for nutrients added to growth media)
	for rxn in model.reactions:
		if rxn in model.exchanges:
			rxn.lower_bound = -RBA_defaults.vmax if float(rxn.lower_bound) == 0 else rxn.lower_bound
			rxn.upper_bound = RBA_defaults.vmax
	
	# input any additional constraints here
	## Xylose conditions
	m.reactions.EX_glc__D_e.lower_bound = -1000
	m.reactions.EX_xyl__D_e.lower_bound = -1000
	m.reactions.EX_xylt_e.lower_bound = -1000  
	m.reactions.EX_abt__L_e.lower_bound = -1000
	
	fva = cobra.flux_analysis.variability.flux_variability_analysis(m, fraction_of_optimum=0)
	# make list of blocked reactions from FVA results
	blocked_rxns = [rxn for rxn, row in fva.iterrows() if row['minimum'] >= -flux_tolerance and row['maximum'] <= flux_tolerance]
	# update rxn reversibility based on FVA results (e.g., if rxn is reversible but FVA shows it can only go in one direction, make it irreversible)
	## this helps reduce the scope of the model further, improving curation and solving time, since reversible rxns are treated as 2 rxns in RBA
	new_bounds = {}
	for rxn, row in fva.iterrows():
		if rxn in blocked_rxns or rxn not in original_bounds:
			continue
		if model.reactions.get_by_id(rxn).lower_bound < 0 and row['minimum'] >= -flux_tolerance:
			new_bounds[rxn] = (0, original_bounds[rxn][1])
		elif model.reactions.get_by_id(rxn).upper_bound > 0 and row['maximum'] <= flux_tolerance:
			new_bounds[rxn] = (original_bounds[rxn][0], 0)

for rxn in model.reactions.copy():
	if rxn.id in blocked_rxns:
		print(f'Removed blocked reaction {rxn.id}')
		model.remove_reactions([rxn], remove_orphans=remove_orphans)
if reduce_reversible_rxns:
	for rxn, bounds in new_bounds.items():
		if rxn in model.reactions:
			if model.reactions.get_by_id(rxn).bounds != bounds:
				print(f'Updated reaction {rxn} bounds from {model.reactions.get_by_id(rxn).bounds} to {bounds} based on FVA results')
				model.reactions.get_by_id(rxn).bounds = bounds

Removed blocked reaction TRDRDt_c_x
Removed blocked reaction FALDH_c
Removed blocked reaction ACCAH_x
Removed blocked reaction FBP2_c
Removed blocked reaction LYSTRS_m
Removed blocked reaction ZYM181t_l_rm
Removed blocked reaction EAR80y_m
Removed blocked reaction EAR60y_m
Removed blocked reaction HISTRS_m
Removed blocked reaction PNP_c
Removed blocked reaction FECPOR_m
Removed blocked reaction NADK_m
Removed blocked reaction BHMT_n
Removed blocked reaction ALCD19y_c
Removed blocked reaction OHPHM_m
Removed blocked reaction DOLK_c
Removed blocked reaction VALTRS_m
Removed blocked reaction ACHLE3_c
Removed blocked reaction URO_x
Removed blocked reaction 4H2OGLTt_c_x
Removed blocked reaction ZYM161t_l_rm
Removed blocked reaction LANOST181t_l_rm
Removed blocked reaction EPIST161t_l_rm
Removed blocked reaction LANOST161t_l_rm
Removed blocked reaction FECOST181t_l_rm
Removed blocked reaction FECOST161t_l_rm
Removed blocked reaction EPIST181t_l_rm
Removed blocked reaction GDPPAILMT_rm
Remove

In [8]:
# iteratively remove rxns until no more can be removed
c = 0
while removed_rxns == 0:
	c += 1
	removed_rxns = 0
	removed_mets = 0
	for met in model.metabolites.copy():
		if len(met.reactions) != 0 and len(met.reactions) < min_rxns_per_met:
			print(f'Removed reaction {met.reactions} using metabolite {met}')
			model.remove_reactions(list(met.reactions)[0], remove_orphans=remove_orphans)
			# model.metabolites.remove(met)
			removed_rxns += 1
			removed_mets += 1
		elif len(met.reactions) == 0:
			# remove metabolite with no reactions
			print(f'Removed metabolite {met} with no reactions')
			model.remove_metabolites([met])
			removed_mets += 1
	# if no rxns were removed on 1st iteration, break the loop
	if removed_rxns == 0 and c == 1:
		print('No reactions removed on 1st iteration, breaking loop')
		break

No reactions removed on 1st iteration, breaking loop


### save model

In [9]:
# replace the extension with "_reduced" in the name, using regex
save_cobra_model(model, '_reduced'.join(os.path.splitext(SM_path)))